In [2]:
pip install senticnet pandas scikit-learn openpyxl emoji contractions


   ---------- ----------------------------- 1/4 [anyascii]
   ---------- ----------------------------- 1/4 [anyascii]
   ---------------------------------------- 4/4 [contractions]

Note: you may need to restart the kernel to use updated packages.


### Normalisation functions

In [42]:
import re
import string
import emoji
import contractions

def remove_urls(text):
    return re.sub(r'http\S+|www\S+', '', text)

def remove_mentions(text):
    return re.sub(r'@\w+', '', text)

def normalise_hashtags(text):
    # #happy -> happy
    return re.sub(r'#(\w+)', r'\1', text)

def normalise_emoji(text):
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = text.replace("_", " ")
    return text

def expand_contractions(text):
    # can't -> cannot, it's -> it is
    return contractions.fix(text)

def normalise_elongated(text):
    # soo -> so, goooood -> good-ish
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

def clean_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

def normalise_microtext(text):
    text = str(text)

    text = text.lower()

    text = remove_urls(text)
    text = remove_mentions(text)
    text = normalise_hashtags(text)
    text = normalise_emoji(text)
    text = expand_contractions(text)
    text = normalise_elongated(text)
    text = remove_punctuation(text)
    text = clean_whitespace(text)

    return text

### Classification & Evaluation

In [43]:
import pandas as pd
import time
from senticnet.senticnet import SenticNet
from sklearn.metrics import classification_report

sn = SenticNet()
 
def classify_with_senticnet(text):
    """
    Returns (subjectivity_label, polarity_label)
      Subjectivity : 0 = Objective,  1 = Subjective
      Polarity     : 0 = Negative,   1 = Positive
    """
    normalised_text = normalise_microtext(text)
    words = normalised_text.split()
    total_polarity = 0
    match_count = 0
 
    for word in words:
        try:
            val = float(sn.polarity_value(word))
            total_polarity += val
            match_count += 1
        except KeyError:
            continue  # word not in knowledge base
 
    subjectivity = 1 if match_count > 0 else 0
    polarity     = 1 if total_polarity > 0 else 0
 
    return subjectivity, polarity

In [44]:
REDDIT_PLACEHOLDERS = {"[removed]", "[deleted]"}
def is_valid_body(text: str) -> bool:
    """
    Returns False for empty strings and Reddit placeholder bodies
    so they are never passed to the classifier.
    """
    stripped = str(text).strip()
    return bool(stripped) and stripped not in REDDIT_PLACEHOLDERS

In [45]:
def extract_comments(comments: list, post_id: str, records: list):
    """
    Recursively walks comments and their replies at any depth,
    appending valid bodies to records.
    """
    for comment in comments:
        cbody = comment.get("body", "")
        if is_valid_body(cbody):
            records.append({
                "id":     post_id,
                "source": "comment",
                "body":   cbody,
            })
        # Recurse into replies regardless of whether this comment had a valid body
        if comment.get("replies"):
            extract_comments(comment["replies"], post_id, records)

In [46]:
import json

def load_bodies_from_json(path: str) -> pd.DataFrame:
    """
    Extracts every usable 'body' from posts and all nested comments/replies.
    For posts whose body is empty or a placeholder, falls back to 'description'.
    Returns a DataFrame with columns: id, source, body
    """
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
 
    records = []
    skipped_posts = 0
 
    for post in data["posts"]:
        post_id = post.get("id", "")
 
        # Post body — fall back to description if body is empty/placeholder
        body = post.get("body", "")
        if is_valid_body(body):
            records.append({"id": post_id, "source": "post", "body": body})
        else:
            description = post.get("description", "")
            if is_valid_body(description):
                records.append({"id": post_id, "source": "post", "body": description})
            else:
                skipped_posts += 1  # link/image post with no text at all
 
        # Comments — recursive to capture all reply levels
        extract_comments(post.get("comments", []), post_id, records)
 
    if skipped_posts:
        print(f"  Note: {skipped_posts} posts had no usable text in body or description and were skipped.")
 
    return pd.DataFrame(records)

In [47]:
def run_evaluation(json_path: str = "enriched_results.json"):
    print("Loading records...")
    df = load_bodies_from_json(json_path)
    counts = df['source'].value_counts().to_dict()
    print(f"Loaded {len(df)} records  (posts: {counts.get('post', 0)}, comments: {counts.get('comment', 0)})\n")
 
    subj_results = []
    pol_results  = []
 
    start_time = time.time()
 
    for body in df["body"]:
        subj, pol = classify_with_senticnet(body)
        subj_results.append(subj)
        pol_results.append(pol)
 
    end_time = time.time()
 
    df["pred_subjectivity"] = subj_results
    df["pred_polarity"]     = pol_results
 
    # ── Summary ───────────────────────────────────────────────────────────────
    n         = len(df)
    n_subj    = sum(subj_results)
    n_obj     = n - n_subj
    n_pos     = sum(pol_results)
    n_neg     = n_subj - n_pos   # polarity only meaningful on subjective records
 
    duration        = end_time - start_time
    records_per_sec = n / duration
 
    print("── Subtask 1 : Subjectivity Detection ─────────────────────────")
    print(f"  Subjective : {n_subj:>6}  ({n_subj/n*100:.1f}%)")
    print(f"  Objective  : {n_obj:>6}  ({n_obj/n*100:.1f}%)")
 
    print("\n── Subtask 2 : Polarity Detection (subjective records only) ───")
    print(f"  Positive   : {n_pos:>6}  ({n_pos/max(n_subj,1)*100:.1f}%)")
    print(f"  Negative   : {n_neg:>6}  ({n_neg/max(n_subj,1)*100:.1f}%)")
 
    print(f"\n── Speed ───────────────────────────────────────────────────────")
    print(f"  {n} records in {duration:.2f}s  →  {records_per_sec:.2f} records/second")
 
    # ── Save results ──────────────────────────────────────────────────────────
    out_path = "senticnet_results.csv"
    df.to_csv(out_path, index=False)
    print(f"\nResults saved to: {out_path}")
 
    return df

In [48]:
run_evaluation("enriched_results.json")

Loading records...
  Note: 62 posts had no usable text in body or description and were skipped.
Loaded 13973 records  (posts: 38, comments: 13935)

── Subtask 1 : Subjectivity Detection ─────────────────────────
  Subjective :  12324  (88.2%)
  Objective  :   1649  (11.8%)

── Subtask 2 : Polarity Detection (subjective records only) ───
  Positive   :   9351  (75.9%)
  Negative   :   2973  (24.1%)

── Speed ───────────────────────────────────────────────────────
  13973 records in 2.58s  →  5416.08 records/second

Results saved to: senticnet_results.csv


,id,source,body,pred_subjectivity,pred_polarity
0,1qpq3cc,post,I'm sorry but it has to come out.\n\nWe are ex...,1,1
1,1qpq3cc,comment,It'll keep us all in job - someone has to fix ...,1,0
2,1qpq3cc,comment,Man I don't look forward to digging through th...,1,0
3,1qpq3cc,comment,"As dumb as LLMs can be, they can't match the d...",1,0
4,1qpq3cc,comment,Acting like LLM's dont have examples of terrib...,1,0
...,...,...,...,...,...
13968,1osvp7o,comment,"Sorry /u/Basic-Capital5450, your submission ha...",1,1
13969,1osvp7o,comment,Intrested,0,0
13970,1osvp7o,comment,"Sorry /u/Low-Necessary2673, your submission ha...",1,1
13971,1osvp7o,comment,"Sorry /u/Mercer1198, your submission has been ...",1,1
